In [2]:
!pip install openvino-dev onnx torchvision pillow

In [5]:
!pip uninstall -y numpy
!pip install numpy==1.23.5
!pip install torch torchvision --upgrade --force-reinstall

Found existing installation: numpy 2.4.3
Uninstalling numpy-2.4.3:
  Successfully uninstalled numpy-2.4.3
  Using cached numpy-1.23.5.tar.gz (10.7 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-an

In [1]:
import torch
import torchvision.models as models
import numpy as np
import time
from PIL import Image
import torchvision.transforms as transforms

In [4]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 14.2 MB/s eta 0:00:00


In [5]:
# Load ResNet18 (pretrained)
model = models.resnet18(pretrained=True)
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [6]:
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    "resnet18.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11
)

print("ONNX model created")

W0328 14:53:04.570000 10293 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 40 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
ONNX model created


In [7]:
from openvino.tools.mo import convert_model

ov_model = convert_model("resnet18.onnx")

print("Converted to OpenVINO IR format")

[ INFO ] MO command line tool is considered as the legacy conversion API as of OpenVINO 2023.2 release.
In 2025.0 MO command line tool and openvino.tools.mo.convert_model() will be removed. Please use OpenVINO Model Converter (OVC) or openvino.convert_model(). OVC represents a lightweight alternative of MO and provides simplified model conversion API. 
Find more information about transition from MO to OVC at https://docs.openvino.ai/2023.2/openvino_docs_OV_Converter_UG_prepare_model_convert_model_MO_OVC_transition.html
Converted to OpenVINO IR format


In [8]:
from openvino.runtime import Core

core = Core()
compiled_model = core.compile_model(ov_model, "CPU")

print("OpenVINO model loaded")

OpenVINO model loaded


In [9]:
from google.colab import files

uploaded = files.upload()

Saving WhatsApp Image 2026-03-19 at 23.47.12 (1).jpeg to WhatsApp Image 2026-03-19 at 23.47.12 (1).jpeg


In [10]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

image_path = list(uploaded.keys())[0]

img = Image.open(image_path).convert("RGB")
img = transform(img).unsqueeze(0)

print("Image loaded and processed")

Image loaded and processed


In [11]:
start = time.time()

with torch.no_grad():
    output_pt = model(img)

end = time.time()

print("PyTorch Inference Time:", end - start)

PyTorch Inference Time: 0.11177372932434082


In [12]:
input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)

img_np = img.numpy()

start = time.time()

result = compiled_model([img_np])[output_layer]

end = time.time()

print("OpenVINO Inference Time:", end - start)

OpenVINO Inference Time: 0.08136773109436035


In [13]:
pred_class = np.argmax(result)
print("Predicted Class Index:", pred_class)

Predicted Class Index: 457
